# Лабораторна робота №2. Частина 2
## Наука про дані: підготовчий етап
### Імпорт модулів та завантаження й очищення датасету

In [1]:
import os
import timeit
import numpy as np
import pandas as pd

# Шлях до завантаженого файлу датасету
DATASET_PATH = "household_power_consumption.txt"

def load_and_clean_data(filepath):
    # Зчитування датасету з розділювачем ';'
    df = pd.read_csv(filepath, sep=';', low_memory=False)
    
    # Заміна символу '?' на NaN для подальшого очищення
    df = df.replace('?', np.nan)
    
    # Визначення числових стовпців для конвертації
    numeric_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage', 
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    
    # Перетворення типів даних у числовий формат
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # Видалення рядків, що містять пропущені значення (NaN) у числових стовпцях
    df = df.dropna(subset=numeric_cols)
    
    return df

# Оцінка часу виконання завантаження та очищення даних за допомогою модуля timeit
start_time = timeit.default_timer()
main_df = load_and_clean_data(DATASET_PATH)
elapsed_time = timeit.default_timer() - start_time

print(f"Загальна кількість записів після очищення: {len(main_df)}")
print(f"Час виконання процедури очищення: {elapsed_time:.4f} секунд")

Загальна кількість записів після очищення: 2049280
Час виконання процедури очищення: 15.0904 секунд


### Завдання 1:
Обрати всі записи, у яких загальна активна споживана потужність перевищує $5\text{ кВт}$.

In [2]:
def select_power_exceeding_5(df):
    return df[df['Global_active_power'] > 5]

start_time = timeit.default_timer()
result_task1 = select_power_exceeding_5(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Кількість знайдених записів: {len(result_task1)}")
print(f"Час виконання процедури: {elapsed_time:.4f} секунд")
if not result_task1.empty:
    display(result_task1.head())

Кількість знайдених записів: 17547
Час виконання процедури: 0.0723 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


### Завдання 2:
Обрати всі записи, у яких сила струму лежить в межах $19\text{--}20\text{ А}$, для них виявити ті, у яких пральна машина та холодильник споживають більше, ніж бойлер та кондиціонер.

In [3]:
def select_intensity_and_submetering(df):
    # Фільтрація за силою струму в межах 19-20 А та порівняння груп споживання 2 і 3
    condition = (df['Global_intensity'] >= 19) & \
                (df['Global_intensity'] <= 20) & \
                (df['Sub_metering_2'] > df['Sub_metering_3'])
    return df[condition]

start_time = timeit.default_timer()
result_task2 = select_intensity_and_submetering(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Кількість знайдених записів: {len(result_task2)}")
print(f"Час виконання процедури: {elapsed_time:.4f} секунд")
if not result_task2.empty:
    display(result_task2.head())

Кількість знайдених записів: 2509
Час виконання процедури: 0.0224 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


### Завдання 3:
Обрати випадковим чином $500\text{ }000$ записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [4]:
def random_sample_and_means(df, sample_size=500000):
    if len(df) < sample_size:
        sample_df = df.sample(n=len(df), replace=False)
    else:
        sample_df = df.sample(n=sample_size, replace=False)
        
    means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

start_time = timeit.default_timer()
result_task3 = random_sample_and_means(main_df)
elapsed_time = timeit.default_timer() - start_time

print("Середні величини для 3-х груп споживання:")
print(result_task3)
print(f"Час виконання процедури: {elapsed_time:.4f} секунд")

Середні величини для 3-х груп споживання:
Sub_metering_1    1.117874
Sub_metering_2    1.289168
Sub_metering_3    6.454434
dtype: float64
Час виконання процедури: 0.3419 секунд


### Завдання 4:
Обрати ті записи, які після 18-00 споживають понад $6\text{ кВт}$ за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [5]:
def complex_selection_and_slicing(df):
    # Фільтрація за часом, потужністю та переважанням групи споживання 2
    condition = (df['Time'] > '18:00:00') & \
                (df['Global_active_power'] > 6) & \
                (df['Sub_metering_2'] > df['Sub_metering_1']) & \
                (df['Sub_metering_2'] > df['Sub_metering_3'])
    
    filtered_df = df[condition]
    
    if filtered_df.empty:
        return pd.DataFrame()
        
    half_index = len(filtered_df) // 2
    
    # Вибірка кожного третього елемента з першої половини
    first_half = filtered_df.iloc[:half_index:3]
    # Вибірка кожного четвертого елемента з другої половини
    second_half = filtered_df.iloc[half_index::4]
    
    return pd.concat([first_half, second_half])

start_time = timeit.default_timer()
result_task4 = complex_selection_and_slicing(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Кількість отриманих записів після кроку зрізання: {len(result_task4)}")
print(f"Час виконання процедури: {elapsed_time:.4f} секунд")
if not result_task4.empty:
    display(result_task4.head())

Кількість отриманих записів після кроку зрізання: 310
Час виконання процедури: 0.0545 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


### Завдання 5:
Пронормувати та стандартизувати вибраний датасет.

In [6]:
def normalize_and_standardize(df):
    numeric_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage', 
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    
    df_norm = df.copy()
    df_std = df.copy()
    
    # Нормалізація (Min-Max Scaling)
    df_norm[numeric_cols] = (df[numeric_cols] - df[numeric_cols].min()) / (df[numeric_cols].max() - df[numeric_cols].min())
    
    # Стандартизація (Z-score Normalization)
    df_std[numeric_cols] = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()
    
    return df_norm, df_std

start_time = timeit.default_timer()
normalized_df, standardized_df = normalize_and_standardize(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Час виконання нормалізації та стандартизації: {elapsed_time:.4f} секунд")
print("Приклад нормованого датасету:")
display(normalized_df.head(2))
print("Приклад стандартизованого датасету:")
display(standardized_df.head(2))

Час виконання нормалізації та стандартизації: 4.0437 секунд
Приклад нормованого датасету:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,16/12/2006,17:25:00,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129


Приклад стандартизованого датасету:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,2.955076,2.610720,-1.851816,3.098788,-0.182337,-0.051274,1.249420
1,16/12/2006,17:25:00,4.037084,2.770405,-2.225274,4.133799,-0.182337,-0.051274,1.130897


### Завдання 6:
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [7]:
def calculate_correlations(df, col1='Global_active_power', col2='Global_intensity'):
    pearson_corr = df[col1].corr(df[col2], method='pearson')
    spearman_corr = df[col1].corr(df[col2], method='spearman')
    return pearson_corr, spearman_corr

start_time = timeit.default_timer()
pearson, spearman = calculate_correlations(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Коефіцієнт кореляції Пірсона: {pearson:.6f}")
print(f"Коефіцієнт кореляції Спірмена: {spearman:.6f}")
print(f"Час виконання процедури розрахунку кореляцій: {elapsed_time:.4f} секунд")

Коефіцієнт кореляції Пірсона: 0.998889
Коефіцієнт кореляції Спірмена: 0.995372
Час виконання процедури розрахунку кореляцій: 3.0012 секунд


### Завдання 7:
Провести One Hot Encoding категоріального атрибута.

In [8]:
def perform_one_hot_encoding(df):
    # Створення штучного категоріального атрибуту на основі рівнів споживання для демонстрації
    df_with_cat = df.copy()
    df_with_cat['Power_Class'] = pd.cut(
        df_with_cat['Global_active_power'], 
        bins=[0, 1.5, 4.0, np.inf], 
        labels=['Low', 'Medium', 'High']
    )
    
    # Виконання One Hot Encoding для створеного атрибуту
    encoded_df = pd.get_dummies(df_with_cat, columns=['Power_Class'], dtype=int)
    return encoded_df

start_time = timeit.default_timer()
result_task7 = perform_one_hot_encoding(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Час виконання процедури One Hot Encoding: {elapsed_time:.4f} секунд")
display(result_task7.head())

Час виконання процедури One Hot Encoding: 0.7660 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Power_Class_Low,Power_Class_Medium,Power_Class_High
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,0,0,1
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,0,0,1
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,0,0,1
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,0,0,1
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,0,1,0
